In [16]:
%load_ext autoreload
%autoreload 2

import torch
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.transform import Rotation as R

import os 
import sys
root_dir = os.path.abspath('..')
if root_dir not in sys.path:
    sys.path.append(root_dir)

from src.models.utils import hamilton_product, q_conjugate



class MockSystem:
    def __init__(self, poses_list, times_list, predict_func):
        """
        poses_list: lista np.array lub tensorów [x, y, z, qx, qy, qz, qw]
                   kolejność: [t-2, t-1] (dwa znane punkty)
        """
        self.buff_size = 10
        self.frame_n = 2  # Chcemy przewidzieć indeks 2 na podstawie 0 i 1
        self.motion_model = 'LINEAR'
        
        # Inicjalizacja bufora (wypełniamy zerami, potem wstawiamy dane testowe)
        self.poses = torch.zeros((self.buff_size, 7))
        self.time = torch.zeros(self.buff_size)
        
        # Wstawiamy dane historyczne
        # Indeks 0 -> t-2 (najstarszy)
        # Indeks 1 -> t-1 (ostatni znany)
        # Indeks 2 -> tu trafi wynik (t0)
        
        # Uwaga: Twoja funkcja liczy indeksy wstecz modulo buff_size.
        # Przy frame_n=2: 
        # k_idx = 2 (cel)
        # k1_idx = 1 (t-1)
        # k2_idx = 0 (t-2)
        
        self.poses[0] = torch.tensor(poses_list[0], dtype=torch.float32)
        self.poses[1] = torch.tensor(poses_list[1], dtype=torch.float32)
        
        self.time[0] = times_list[0]
        self.time[1] = times_list[1]
        self.time[2] = times_list[2] # Czas dla którego robimy predykcję
        
        self._approx_movement_method = predict_func

    def _approx_movement(self):
        # Wywołujemy Twoją funkcję przekazaną w __init__
        # (bindujemy ją do 'self' tej klasy)
        return self._approx_movement_method(self)

# --- 2. Funkcja Wizualizująca ---
def plot_test_case(title, p_prev, p_last, p_pred, p_expected=None):
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(111, projection='3d')
    
    # Funkcja rysująca układ współrzędnych (quiver)
    def draw_frame(pose, color_label, length=0.5):
        pos = pose[:3].numpy()
        quat = pose[3:].numpy() # x, y, z, w
        
        # Scipy rotacja (dla rysowania osi)
        r = R.from_quat(quat) 
        # Macierz rotacji 3x3
        rot_mat = r.as_matrix()
        
        # Rysujemy osie X (czerwona), Y (zielona), Z (niebieska)
        ax.quiver(pos[0], pos[1], pos[2], rot_mat[0,0], rot_mat[1,0], rot_mat[2,0], color='r', length=length, normalize=True)
        ax.quiver(pos[0], pos[1], pos[2], rot_mat[0,1], rot_mat[1,1], rot_mat[2,1], color='g', length=length, normalize=True)
        ax.quiver(pos[0], pos[1], pos[2], rot_mat[0,2], rot_mat[1,2], rot_mat[2,2], color='b', length=length, normalize=True)
        
        ax.scatter(pos[0], pos[1], pos[2], label=color_label, s=50)

    # Rysowanie punktów
    draw_frame(p_prev, 't-2 (Start)')
    draw_frame(p_last, 't-1 (Ostatni znany)')
    draw_frame(p_pred, 'PREDYKCJA', length=0.8)
    
    if p_expected is not None:
        # Rysujemy expected jako "duch" (mniejsza przezroczystość/inny styl)
        pos = p_expected[:3].numpy()
        ax.scatter(pos[0], pos[1], pos[2], c='k', marker='x', s=100, label='OCZEKIWANY')

    # Rysowanie linii trajektorii
    path = np.array([p_prev[:3].numpy(), p_last[:3].numpy(), p_pred[:3].numpy()])
    ax.plot(path[:,0], path[:,1], path[:,2], 'k--', alpha=0.5)

    ax.set_title(title)
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.legend()
    
    # Ustawienie limitów żeby wykres był czytelny (axis equal hack)
    all_pts = np.vstack([p_prev[:3], p_last[:3], p_pred[:3]])
    mid_x, mid_y, mid_z = np.mean(all_pts, axis=0)
    ax.set_xlim(mid_x - 1, mid_x + 1)
    ax.set_ylim(mid_y - 1, mid_y + 1)
    ax.set_zlim(mid_z - 1, mid_z + 1)
    
    plt.show()

# --- 3. Funkcja uruchamiająca test ---
def run_test(test_name, pose_t2, pose_t1, dt1, dt2, expected_pose, user_function):
    print(f"--- TEST: {test_name} ---")
    
    # Czasy: t-2 = 0, t-1 = dt1, t0 = dt1 + dt2
    times = [0.0, dt1, dt1 + dt2]
    
    # Setup
    mock = MockSystem([pose_t2, pose_t1], times, user_function)
    
    # Uruchomienie Twojej funkcji
    predicted_pose = mock._approx_movement()
    
    # Obliczenie błędów
    pos_err = torch.norm(predicted_pose[:3] - expected_pose[:3])
    
    # Błąd rotacji (kąt między kwaterionami)
    q_pred = predicted_pose[3:]
    q_true = expected_pose[3:]
    
    # Dot product i clamp na wszelki wypadek
    dot = torch.abs(torch.sum(q_pred * q_true))
    dot = torch.clamp(dot, -1.0, 1.0)
    angle_err_deg = 2 * torch.acos(dot) * (180.0 / np.pi)

    print(f"Błąd Pozycji: {pos_err:.4f} m")
    print(f"Błąd Rotacji: {angle_err_deg:.4f} stopni")
    
    status = "SUKCES" if pos_err < 0.01 and angle_err_deg < 1.0 else "UWAGA / BŁĄD"
    print(f"Wynik: {status}")
    
    plot_test_case(test_name, mock.poses[0], mock.poses[1], predicted_pose, expected_pose)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:

def approx_movement(self):
    # indexes
    k_idx = self.frame_n % self.buff_size
    k1_idx = (k_idx - 1) % self.buff_size  
    k2_idx = (k_idx - 2) % self.buff_size

    # get time stams
    t0 = self.time[k_idx]
    t1 = self.time[k1_idx]
    t2 = self.time[k2_idx]

    # get previous position
    x1 = self.poses[k1_idx, :]
    x2 = self.poses[k2_idx, :]

    if self.motion_model == 'LINEAR':

      # linear displacement
      new_pose = x1[0:3] + (x1[0:3] - x2[0:3])/(t1 - t2)*(t0 - t1) 

      # quaterions
      q1 = x1[3:]
      q2 = x2[3:]

      # 1. idk dlaczego tak sie robi 
      dot = (q1 * q2).sum() # element wise
      if dot < 0: q1 = -q1

      # rotation - quaterions difference
      q2_conj = q_conjugate(q2)
      diff = hamilton_product(q1, q2_conj)  # diff q2 -> q1: diff = q2 * q1^-1

      # rotation axis 
      s = torch.sqrt(torch.clamp(1 - diff[-1]*diff[-1], 0.0))
      if s < 0.001: rot_axis = torch.tensor([1, 0, 0], device = diff.device, dtype = diff.dtype)
      else: rot_axis = diff[:-1]/s

      # rotation angle
      rot_angle = 2 * torch.acos(torch.clamp(diff[-1], -1.0, 1.0))
      rot_angle_appro = rot_angle/(t1 - t2)*(t0 - t1) # approximation apriori, to t0

      # apply rotation
      q_step_vect = rot_axis * torch.sin(rot_angle_appro/2)
      q_step_scal = torch.cos(rot_angle_appro/2).unsqueeze(0)   
      q_step = torch.cat((q_step_vect, q_step_scal), dim=0)


      q0 = hamilton_product(q_step, q1)
      q0 = q0 /torch.norm(q0)

      x0 = torch.cat((new_pose, q0), dim=0)
      # self.poses[k_idx, :] = torch.cat((new_pose, q0), dim=0)

    else:
      x0 = x1
      # self.poses[k_idx, :] = x1

    self.poses[k_idx, :] = x0
    return x0




In [18]:
# --- Przygotowanie danych testowych (quaternions x,y,z,w) ---

# TEST 1: Prosta translacja w X (0 -> 1 -> 2), bez rotacji
t1_p_t2 = [0, 0, 0, 0, 0, 0, 1] # x=0
t1_p_t1 = [1, 0, 0, 0, 0, 0, 1] # x=1
t1_exp  = torch.tensor([2, 0, 0, 0, 0, 0, 1], dtype=torch.float32) # Oczekujemy x=2

# TEST 2: Czysta rotacja wokół Z o 90 stopni (PI/2)
# t-2: 0 stopni
# t-1: 90 stopni
# Oczekiwane t0: 180 stopni
q_0 = R.from_euler('z', 0, degrees=True).as_quat()
q_90 = R.from_euler('z', 90, degrees=True).as_quat()
q_180 = R.from_euler('z', 180, degrees=True).as_quat()

t2_p_t2 = np.concatenate([[0,0,0], q_0])
t2_p_t1 = np.concatenate([[0,0,0], q_90])
t2_exp  = torch.tensor(np.concatenate([[0,0,0], q_180]), dtype=torch.float32)

# TEST 3: Spiralny ruch (Translacja Z + Rotacja Z)
# Wznosi się o 1m i obraca o 45 stopni co krok
q_45 = R.from_euler('z', 45, degrees=True).as_quat()
q_90 = R.from_euler('z', 90, degrees=True).as_quat()

t3_p_t2 = np.concatenate([[0,0,0], q_0])   # z=0, deg=0
t3_p_t1 = np.concatenate([[0,0,1], q_45])  # z=1, deg=45
# Oczekujemy z=2, deg=90
t3_exp  = torch.tensor(np.concatenate([[0,0,2], q_90]), dtype=torch.float32)

# --- URUCHOMIENIE TESTÓW ---

# Tutaj podmieniamy 'YourClass._approx_movement' na Twoją funkcję.
# Zakładam, że skopiowałeś poprawioną funkcję z poprzedniej odpowiedzi i nazwałeś ją `corrected_approx_movement`
# Jeśli funkcja jest metodą klasy, możesz ją wyciągnąć: YourClass._approx_movement

# Przykład uruchomienia (użyj swojej nazwy funkcji zamiast corrected_approx_movement):
# run_test("Translacja X", t1_p_t2, t1_p_t1, 1.0, 1.0, t1_exp, corrected_approx_movement)



# UWAGA: Aby testy zadziałały, musisz przekazać faktyczną funkcję.
# Poniżej zakładam, że zdefiniowałeś funkcję 'corrected_approx_movement' 
# na podstawie poprzedniej odpowiedzi.

try:
    # Test 1
    run_test("1. Czysta Translacja X", t1_p_t2, t1_p_t1, 1.0, 1.0, t1_exp, approx_movement)
    
    # Test 2
    run_test("2. Czysta Rotacja Z (90->180)", t2_p_t2, t2_p_t1, 1.0, 1.0, t2_exp, approx_movement)
    
    # Test 3
    run_test("3. Spirala (Ruch Z + Rotacja Z)", t3_p_t2, t3_p_t1, 1.0, 1.0, t3_exp, approx_movement)

except NameError:
    print("BŁĄD: Nie zdefiniowano funkcji 'corrected_approx_movement'. Wklej kod funkcji przed uruchomieniem testów.")

--- TEST: 1. Czysta Translacja X ---


RuntimeError: The size of tensor a (0) must match the size of tensor b (4) at non-singleton dimension 0